In [ ]:
from common.llm_client import create_llm_client
from lcb_runner.benchmarks import code_generation

dataset = code_generation.load_code_generation_dataset(release_version="release_v6", start_date="2024-01-01", diffulty=code_generation.Difficulty.HARD)
sample_problem = dataset[0]
llm_client = create_llm_client(provider="openai", model="gpt-5")

2025-10-27 11:53:26.177 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:143 - Using 'test' split of the dataset.
2025-10-27 11:53:47.603 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:155 - Filtered problems by difficulty: %s
2025-10-27 11:53:47.643 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:165 - Loaded %d problems
2025-10-27 11:53:47.794 | INFO     | common.llm_client:create_llm_client:371 - Creating LLM client: provider=openai, model=gpt-5
2025-10-27 11:53:47.795 | DEBUG    | common.llm_client:__init__:43 - Initialized LLM client: model=gpt-5, max_tokens=1000000, limit_enabled=True
2025-10-27 11:53:47.800 | DEBUG    | common.llm_client:__init__:247 - Initialized OpenAIClient with model: gpt-5, reasoning_effort: minimal
2025-10-27 11:53:47.800 | INFO     | common.llm_client:create_llm_client:403 - Successfully created OpenAIClient


In [24]:
import importlib
from common.coevolution.population import TestPopulation
from common.code_preprocessing.analyzers import extract_test_methods_code
from common.coevolution.bayesian import initialize_prior_beliefs
import common.coevolution.operators as operators

test_operator = operators.TestOperator(llm_client=llm_client, problem=sample_problem)

test_class_block = test_operator.create_initial_population(5)
test_individuals = extract_test_methods_code(test_class_block)

# Initialize prior probabilities
test_probs = initialize_prior_beliefs(
    len(test_individuals), 0.6
)

# Create population
test_population = TestPopulation(
    individuals=test_individuals,
    probabilities=test_probs,
    generation=0,
    test_class_block=test_class_block,
)

test_population

2025-10-27 11:54:25.725 | INFO     | common.coevolution.operators:create_initial_population:852 - TestOperator: Creating initial test population (requesting 5 test methods)
2025-10-27 11:54:25.726 | DEBUG    | common.coevolution.operators:_generate_with_retry:379 - Sending prompt to LLM
2025-10-27 11:54:25.726 | DEBUG    | common.llm_client:generate:256 - OpenAIClient generating with model: gpt-5, reasoning_effort: minimal
2025-10-27 11:54:36.332 | DEBUG    | common.llm_client:generate:276 - Generated 2174 characters
2025-10-27 11:54:36.332 | DEBUG    | common.coevolution.operators:_generate_with_retry:388 - Received response from LLM
2025-10-27 11:54:36.333 | INFO     | common.coevolution.operators:create_initial_population:872 - TestOperator: Successfully generated unittest class with 5 test methods
2025-10-27 11:54:36.333 | DEBUG    | common.coevolution.bayesian:initialize_prior_beliefs:225 - Initializing 5 prior beliefs with probability 0.6000
2025-10-27 11:54:36.334 | DEBUG    | c

TestPopulation(size=5, generation=0, avg_prob=0.6000, has_test_class=True)

In [25]:
for test in test_population.individuals:
    print(test)

def test_single_char_k1(self):
    s, k = ('a', 1)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 1)
def test_all_same_chars_k1_long(self):
    s, k = ('aaaaa', 1)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 5)
def test_two_distinct_k2_mixed(self):
    s, k = ('abcaab', 2)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 4)
def test_full_alphabet_k26(self):
    s, k = ('abcdefghijklmnopqrstuvwxyz', 26)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 1)
def test_alternating_three_chars_k1(self):
    s, k = ('abcabc', 1)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 6)


In [26]:
import random
# after editing operators.py on disk:
operators = importlib.reload(operators)    # re-executes the module
# rebind the class from the (new) module
TestOperator = operators.TestOperator


test_operator = TestOperator(llm_client=llm_client, problem=sample_problem)
print("Crossover Result:")
parent_1 = random.choice(test_population.individuals)
parent_2 = random.choice(test_population.individuals)
print("Parent 1:")
print(parent_1)
print("Parent 2:")
print(parent_2)

print(test_operator.crossover(parent_1, parent_2))

print("Mutation Result:")
print(test_operator.mutate(parent_1))

2025-10-27 11:54:42.066 | INFO     | common.coevolution.operators:crossover:923 - TestOperator: Performing crossover on two test cases
2025-10-27 11:54:42.067 | DEBUG    | common.coevolution.operators:_generate_with_retry:379 - Sending prompt to LLM
2025-10-27 11:54:42.067 | DEBUG    | common.llm_client:generate:256 - OpenAIClient generating with model: gpt-5, reasoning_effort: minimal


Crossover Result:
Parent 1:
def test_two_distinct_k2_mixed(self):
    s, k = ('abcaab', 2)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 4)
Parent 2:
def test_all_same_chars_k1_long(self):
    s, k = ('aaaaa', 1)
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 5)


2025-10-27 11:54:45.704 | DEBUG    | common.llm_client:generate:276 - Generated 504 characters
2025-10-27 11:54:45.704 | DEBUG    | common.coevolution.operators:_generate_with_retry:388 - Received response from LLM
2025-10-27 11:54:45.705 | INFO     | common.coevolution.operators:crossover:948 - TestOperator: Crossover completed
2025-10-27 11:54:45.705 | INFO     | common.coevolution.operators:mutate:891 - TestOperator: Performing mutation on test case
2025-10-27 11:54:45.705 | DEBUG    | common.coevolution.operators:_generate_with_retry:379 - Sending prompt to LLM
2025-10-27 11:54:45.706 | DEBUG    | common.llm_client:generate:256 - OpenAIClient generating with model: gpt-5, reasoning_effort: minimal


def test_varied_chars_k3_split(self):
    s, k = ('abcabc', 3)
    # With k=3, entire string has exactly 3 distinct chars; without change it forms 1 partition.
    # Change one character (e.g., s[3]='a' -> 'a') to help break earlier?
    # Actually, any single change cannot increase partitions since the full string still has <=3 distinct chars,
    # so longest prefix is the whole string. Maximum partitions remains 1.
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 1)
Mutation Result:


2025-10-27 11:54:48.952 | DEBUG    | common.llm_client:generate:276 - Generated 310 characters
2025-10-27 11:54:48.953 | DEBUG    | common.coevolution.operators:_generate_with_retry:388 - Received response from LLM
2025-10-27 11:54:48.954 | INFO     | common.coevolution.operators:mutate:909 - TestOperator: Mutation completed


def test_single_char_k1_all_same(self):
    s, k = ('aaaaa', 1)
    # With k=1 and all characters the same, the entire string is one partition;
    # changing any one character cannot increase the number of partitions beyond 1.
    self.assertEqual(self.sol.maxPartitionsAfterOperations(s, k), 1)
